In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [32]:
from pathlib import Path


cars_path = Path("cars_split.csv")
if not cars_path.exists():
    cars_path = Path("..") / "cars_split.csv"

df = pd.read_csv(cars_path)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car brand,car model
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet,chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick,skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth,satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc,rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford,torino


In [33]:
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
hp_median = df['horsepower'].median()
df['horsepower'] = df['horsepower'].fillna(hp_median)
print(f"Imputed 6 missing horsepower values with median = {hp_median}")

Imputed 6 missing horsepower values with median = 93.5


In [34]:
df['fuel_efficient'] = (df['mpg'] >= 23).astype(int)
label_counts = df['fuel_efficient'].value_counts()
print(f"\nTarget distribution: 0 (not efficient) = {label_counts[0]}, 1 (efficient) = {label_counts[1]}")


Target distribution: 0 (not efficient) = 197, 1 (efficient) = 201


In [35]:
NUMERIC_FEATURES = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']
CATEGORICAL_FEATURES = ['origin', 'car brand']

df_encoded = pd.get_dummies(df, columns=CATEGORICAL_FEATURES, drop_first=True)


In [36]:
ohe_cols = [c for c in df_encoded.columns
            if c.startswith('origin_') or c.startswith('car brand_')]
feature_cols = NUMERIC_FEATURES + ohe_cols
 
X = df_encoded[feature_cols]
y = df_encoded['fuel_efficient']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test  = X_test.copy()
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_test_scaled[NUMERIC_FEATURES]  = scaler.transform(X_test[NUMERIC_FEATURES])

Train: 318 | Test: 80


In [38]:
model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [39]:
y_pred      = model.predict(X_test_scaled)
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
 
acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_prob)

print("\n── Evaluation Metrics ──────────────────────────────")
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Efficient (0)', 'Efficient (1)']))


── Evaluation Metrics ──────────────────────────────
  Accuracy  : 0.9500
  Precision : 0.9091
  Recall    : 1.0000
  F1 Score  : 0.9524
  ROC-AUC   : 0.9975

Full Classification Report:
                   precision    recall  f1-score   support

Not Efficient (0)       1.00      0.90      0.95        40
    Efficient (1)       0.91      1.00      0.95        40

         accuracy                           0.95        80
        macro avg       0.95      0.95      0.95        80
     weighted avg       0.95      0.95      0.95        80



In [40]:
fig = plt.figure(figsize=(18, 13))
fig.patch.set_facecolor('#0f1117')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)
 
ACCENT   = '#7c6af7'
ACCENT2  = '#f7a26a'
GRID_CLR = '#2a2d3e'
TEXT_CLR = '#e0e0f0'
BG       = '#1a1d2e'
 
def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT_CLR, labelsize=9)
    ax.title.set_color(TEXT_CLR)
    ax.title.set_fontsize(11)
    ax.title.set_fontweight('bold')
    ax.set_title(title)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_CLR)
    ax.yaxis.label.set_color(TEXT_CLR)
    ax.xaxis.label.set_color(TEXT_CLR)
    ax.grid(color=GRID_CLR, linewidth=0.6)

<Figure size 1800x1300 with 0 Axes>

In [41]:
ax0 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(y_test, y_pred)
im = ax0.imshow(cm, cmap='Purples')
ax0.set_xticks([0,1]); ax0.set_yticks([0,1])
ax0.set_xticklabels(['Not Eff.', 'Efficient'], color=TEXT_CLR)
ax0.set_yticklabels(['Not Eff.', 'Efficient'], color=TEXT_CLR)
for i in range(2):
    for j in range(2):
        ax0.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color='white', fontsize=16, fontweight='bold')
ax0.set_xlabel('Predicted', color=TEXT_CLR)
ax0.set_ylabel('Actual', color=TEXT_CLR)
style_ax(ax0, 'Confusion Matrix')
ax0.grid(False)

In [42]:
from sklearn.metrics import roc_curve
ax1 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
ax1.plot(fpr, tpr, color=ACCENT, lw=2, label=f'AUC = {roc_auc:.3f}')
ax1.plot([0,1],[0,1], '--', color='#555', lw=1)
ax1.fill_between(fpr, tpr, alpha=0.15, color=ACCENT)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(facecolor=BG, labelcolor=TEXT_CLR, fontsize=9)
style_ax(ax1, 'ROC Curve')

In [43]:
ax2 = fig.add_subplot(gs[0, 2])
metrics = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1, 'ROC-AUC': roc_auc}
bars = ax2.barh(list(metrics.keys()), list(metrics.values()),
                color=[ACCENT, ACCENT2, '#6af7c8', '#f76a9a', '#f7e16a'], height=0.55)
ax2.set_xlim(0, 1.15)
for bar, val in zip(bars, metrics.values()):
    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', color=TEXT_CLR, fontsize=9, fontweight='bold')
style_ax(ax2, 'Evaluation Metrics')
ax2.set_xlabel('Score')

Text(0.5, 0, 'Score')

In [44]:
ax3 = fig.add_subplot(gs[1, :2])
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': model.coef_[0]})
top = coef_df.reindex(coef_df['coef'].abs().sort_values(ascending=False).index).head(15)
colors = [ACCENT if c > 0 else ACCENT2 for c in top['coef']]
ax3.barh(top['feature'][::-1], top['coef'][::-1], color=colors[::-1], height=0.6)
ax3.axvline(0, color='#aaa', linewidth=0.8, linestyle='--')
style_ax(ax3, 'Top 15 Feature Coefficients (scaled)')
ax3.set_xlabel('Coefficient value  (positive → more likely efficient)')

Text(0.5, 0, 'Coefficient value  (positive → more likely efficient)')

In [45]:
ax4 = fig.add_subplot(gs[1, 2])
probs_0 = y_pred_prob[y_test == 0]
probs_1 = y_pred_prob[y_test == 1]
ax4.hist(probs_0, bins=18, alpha=0.7, color=ACCENT2, label='Not Efficient', density=True)
ax4.hist(probs_1, bins=18, alpha=0.7, color=ACCENT,  label='Efficient',     density=True)
ax4.axvline(0.5, color='white', linestyle='--', linewidth=1, label='Decision boundary')
ax4.set_xlabel('Predicted Probability')
ax4.set_ylabel('Density')
ax4.legend(facecolor=BG, labelcolor=TEXT_CLR, fontsize=8)
style_ax(ax4, 'Predicted Probability Distribution')


In [46]:
fig.suptitle('Logistic Regression — Car Fuel Efficiency (MPG ≥ 23)',
             fontsize=15, fontweight='bold', color=TEXT_CLR, y=0.98)
 
plt.savefig('./logistic_regression_results.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print("\nPlot saved.")


Plot saved.


<Figure size 640x480 with 0 Axes>